# Phase 18D Notebook 01: Materialize Kaggle Readiness\n\nThis notebook performs resumable, hash-verified materialization, canonical subject/initial-MRI selection, approved Phase 4 preprocessing reuse, and deterministic source/target manifests. It may generate model-ready artifacts only inside Kaggle; it performs no model training or predictive evaluation.

In [ ]:
authorization_flags = {"authorized": False, "real_execution_authorized": False, "publication_authorized": False, "phase_19_forbidden": True}
PHASE_19_FORBIDDEN = True
from __future__ import annotations
import csv, hashlib, hmac, json, os
from collections import defaultdict
from pathlib import Path

BINDING = Path('/kaggle/working/acda3d_binding')
OUTPUT = Path('/kaggle/working/acda3d_model_ready/adni')
SECRET = os.environ.get('ACDA3D_SUBJECT_HMAC_KEY') or os.environ.get('KAGGLE_HMAC_SECRET')
if not BINDING.is_dir(): raise RuntimeError('BINDING_EVIDENCE_MISSING: execute Notebook 00 first')
if not SECRET: raise RuntimeError('HMAC_SECRET_MISSING')
PHASE4_CONFIG = os.environ.get('ACDA3D_PHASE4_CONFIG')
if os.environ.get('ACDA3D_PHASE4_RUNTIME_READY', 'false').lower() != 'true' or not PHASE4_CONFIG:
    raise RuntimeError('PHASE4_RUNTIME_NOT_BOUND: configure ACDA3D_PHASE4_CONFIG for the approved repository preprocessing runtime')
metadata = json.loads((BINDING/'metadata_manifest.json').read_text())
metadata_path = Path('/kaggle/input') / metadata['relative_path']
if not metadata_path.is_file(): raise FileNotFoundError('ADNI_METADATA_MISSING_AT_BOUND_PATH')
if hashlib.sha256(metadata_path.read_bytes()).hexdigest() != metadata['sha256']: raise RuntimeError('ADNI_METADATA_HASH_MISMATCH')
with metadata_path.open(encoding='utf-8-sig', newline='') as handle: rows = list(csv.DictReader(handle))
columns = set(rows[0]) if rows else set()
def column(aliases):
    for name in aliases:
        if name in columns: return name
    raise ValueError(f'REQUIRED_METADATA_FIELD_MISSING: {aliases}')
subject_col = column(('subject_id','Subject','subject','RID','PTID'))
diagnosis_col = column(('diagnosis','DX','Diagnosis','label','Group'))
scan_col = column(('scan_id','image_id','ImageID','SeriesID','study_id'))
visit_col = next((x for x in ('visit_code','visit','VISCODE','visit_type') if x in columns), None)
date_col = next((x for x in ('visit_date','scan_date','EXAMDATE','date') if x in columns), None)
groups = defaultdict(list)
for row in rows:
    diagnosis = (row.get(diagnosis_col) or '').strip().upper()
    if diagnosis not in {'CN','MCI','AD'}: raise ValueError('ADNI_METADATA_INVALID_LABEL')
    groups[(row.get(subject_col) or '').strip()].append(row)
def rank(row):
    visit = (row.get(visit_col) or '').strip().lower() if visit_col else ''
    priority = 0 if visit == 'bl' else 1 if visit in {'sc','scmri'} else 2
    return (priority, (row.get(date_col) or '') if date_col else '', (row.get(scan_col) or ''))
def subject_hash(subject): return hmac.new(SECRET.encode(), f'ADNI:{subject}'.encode(), hashlib.sha256).hexdigest()
def sha256(path): return hashlib.sha256(path.read_bytes()).hexdigest()
OUTPUT.mkdir(parents=True, exist_ok=True)
status_path = OUTPUT/'subject_status.json'
statuses = json.loads(status_path.read_text()) if status_path.exists() else {}
manifest_rows = []
for subject, candidates in sorted(groups.items()):
    if not subject: raise ValueError('ADNI_SUBJECT_MISSING')
    ordered = sorted(candidates, key=rank)
    if len(ordered) > 1 and rank(ordered[0]) == rank(ordered[1]):
        statuses[subject] = {'status':'FAILED','reason':'UNRESOLVED_INITIAL_SCAN_TIE'}
        continue
    selected = ordered[0]; token = subject_hash(subject); output_path = OUTPUT/f'{token}.pt'
    state = statuses.get(subject, {'status':'PENDING'})
    if state.get('status') == 'COMPLETED' and output_path.is_file() and sha256(output_path) == state.get('model_ready_sha256'):
        pass
    else:
        statuses[subject] = {'status':'RUNNING','source_scan_id':selected[scan_col], 'selected_visit':selected.get(visit_col) if visit_col else None}
        try:
            from acda3d.data.preprocessing import load_preprocessing_config, run_preprocessing
            phase4_cfg = load_preprocessing_config(PHASE4_CONFIG)
            if phase4_cfg.data.cohort.upper() != 'ADNI':
                raise RuntimeError('PHASE4_CONFIG_COHORT_MISMATCH')
            records = run_preprocessing(phase4_cfg, subjects={subject})
            completed = [record for record in records if record.status.lower() == 'completed' and not record.skipped]
            if len(completed) != 1:
                raise RuntimeError('PHASE4_PREPROCESSING_DID_NOT_PRODUCE_ONE_COMPLETED_ARTIFACT')
            output_path = Path(completed[0].output_path)
        except Exception as exc:
            statuses[subject] = {'status':'FAILED','reason':str(exc)}
            continue
    statuses[subject].update({'status':'COMPLETED','model_ready_sha256':sha256(output_path)})
    manifest_rows.append({'subject_hash':token,'cohort':'ADNI','original_label_name':selected[diagnosis_col], 'binary_label_name':'CN' if selected[diagnosis_col]=='CN' else 'Impaired', 'binary_label':0 if selected[diagnosis_col]=='CN' else 1, 'canonical_source_scan_id_hash':hashlib.sha256(selected[scan_col].encode()).hexdigest(), 'model_ready_relative_path':str(output_path.relative_to(Path('/kaggle/working'))).replace('\','/'), 'model_ready_sha256':sha256(output_path), 'status':'COMPLETED'})
status_path.write_text(json.dumps(statuses, indent=2, sort_keys=True), encoding='utf-8')
(OUTPUT/'adni_binary_manifest.json').write_text(json.dumps({'task_id':'cn_vs_impaired','class_order':['CN','Impaired'],'persons':manifest_rows}, indent=2), encoding='utf-8')
with (OUTPUT/'adni_model_ready_provenance.jsonl').open('w', encoding='utf-8') as handle:
    for row in manifest_rows: handle.write(json.dumps(row, sort_keys=True)+'\n')
print(f'Materialized {len(manifest_rows)} ADNI persons')
